# Water Quality Data Cleaning: EPA Stations

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from shapely.geometry import Point
from pathlib import Path

In [2]:
data_path = r'../../../../data/tabular/water-quality/raw'

df_wq = pd.read_csv(f'{data_path}/epa-wq.csv')

'ls' is not recognized as an internal or external command,
operable program or batch file.
C:\Users\owenm\AppData\Local\Temp\ipykernel_2220\1550114532.py:4: DtypeWarning: Columns (0: ActivityMediaSubdivisionName, 1: ActivityRelativeDepthName, 2: ActivityDepthHeightMeasure/MeasureUnitCode, 3: ActivityConductingOrganizationText, 4: ActivityCommentText, 5: HydrologicCondition, 6: HydrologicEvent, 7: SampleCollectionMethod/MethodDescriptionText, 8: MeasureQualifierCode, 9: StatisticalBaseCode, 10: ResultWeightBasisText, 11: ResultTimeBasisText, 12: ResultTemperatureBasisText, 13: SubjectTaxonomicName, 14: SampleTissueAnatomyName, 15: LaboratoryName, 16: ResultLaboratoryCommentText) have mixed types. Specify dtype option on import or set low_memory=False.
  df_wq = pd.read_csv(f'{data_path}/epa-wq.csv')


## Step 1: Inspect the data

Before making any changes to the data, I must inspect the data to gain a better understanding of it.

In [3]:
df_wq.shape          # How many rows and columns?

(118604, 81)

In [4]:
df_wq.head()        # What does the data look like?

,OrganizationIdentifier,OrganizationFormalName,ActivityIdentifier,ActivityTypeCode,ActivityMediaName,ActivityMediaSubdivisionName,ActivityStartDate,ActivityStartTime/Time,ActivityStartTime/TimeZoneCode,ActivityEndDate,...,LaboratoryName,AnalysisStartDate,ResultLaboratoryCommentText,ResultDetectionQuantitationLimitUrl,DetectionQuantitationLimitTypeName,DetectionQuantitationLimitMeasure/MeasureValue,DetectionQuantitationLimitMeasure/MeasureUnitCode,LabSamplePreparationUrl,LastUpdated,ProviderName
0,USGS-IA,USGS Iowa Water Science Center,nwisia.01.02400517,Sample-Routine,Water,Surface Water,2024-01-01,12:00:00,CST,NaN,...,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,USGS-IA,USGS Iowa Water Science Center,nwisia.01.02400517,Sample-Routine,Water,Surface Water,2024-01-01,12:00:00,CST,NaN,...,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,USGS-IA,USGS Iowa Water Science Center,nwisia.01.02400551,Sample-Routine,Water,Surface Water,2024-02-26,10:30:00,CST,NaN,...,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,USGS-IA,USGS Iowa Water Science Center,nwisia.01.02400551,Sample-Routine,Water,Surface Water,2024-02-26,10:30:00,CST,NaN,...,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
4,USGS-IA,USGS Iowa Water Science Center,nwisia.01.02400551,Sample-Routine,Water,Surface Water,2024-02-26,10:30:00,CST,NaN,...,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS


In [5]:
# What type is each column?
for i, t in enumerate(df_wq.dtypes):
    print(f'{t}: {df_wq.columns[i]}')

str: OrganizationIdentifier
str: OrganizationFormalName
str: ActivityIdentifier
str: ActivityTypeCode
str: ActivityMediaName
str: ActivityMediaSubdivisionName
str: ActivityStartDate
str: ActivityStartTime/Time
str: ActivityStartTime/TimeZoneCode
float64: ActivityEndDate
float64: ActivityEndTime/Time
float64: ActivityEndTime/TimeZoneCode
str: ActivityRelativeDepthName
float64: ActivityDepthHeightMeasure/MeasureValue
str: ActivityDepthHeightMeasure/MeasureUnitCode
float64: ActivityDepthAltitudeReferencePointText
float64: ActivityTopDepthHeightMeasure/MeasureValue
float64: ActivityTopDepthHeightMeasure/MeasureUnitCode
float64: ActivityBottomDepthHeightMeasure/MeasureValue
float64: ActivityBottomDepthHeightMeasure/MeasureUnitCode
str: ProjectIdentifier
str: ProjectName
str: ActivityConductingOrganizationText
str: MonitoringLocationIdentifier
str: MonitoringLocationName
str: ActivityCommentText
float64: SampleAquifer
str: HydrologicCondition
str: HydrologicEvent
float64: ActivityLocation/La

In [6]:
df_wq.info()         # Non-null counts + dtypes at a glance

<class 'pandas.DataFrame'>
RangeIndex: 118604 entries, 0 to 118603
Data columns (total 81 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   OrganizationIdentifier                             118604 non-null  str    
 1   OrganizationFormalName                             118604 non-null  str    
 2   ActivityIdentifier                                 118604 non-null  str    
 3   ActivityTypeCode                                   118604 non-null  str    
 4   ActivityMediaName                                  118604 non-null  str    
 5   ActivityMediaSubdivisionName                       56297 non-null   str    
 6   ActivityStartDate                                  118604 non-null  str    
 7   ActivityStartTime/Time                             117283 non-null  str    
 8   ActivityStartTime/TimeZoneCode                     117283 non-null  str    
 9   Acti

In [7]:
df_wq.describe()     # Summary stats for numeric columns

,ActivityEndDate,ActivityEndTime/Time,ActivityEndTime/TimeZoneCode,ActivityDepthHeightMeasure/MeasureValue,ActivityDepthAltitudeReferencePointText,ActivityTopDepthHeightMeasure/MeasureValue,ActivityTopDepthHeightMeasure/MeasureUnitCode,ActivityBottomDepthHeightMeasure/MeasureValue,ActivityBottomDepthHeightMeasure/MeasureUnitCode,SampleAquifer,...,USGSPCode,ResultDepthHeightMeasure/MeasureValue,ResultDepthHeightMeasure/MeasureUnitCode,ResultDepthAltitudeReferencePointText,BinaryObjectFileName,BinaryObjectFileTypeCode,ResultFileUrl,ResultAnalyticalMethod/MethodUrl,DetectionQuantitationLimitMeasure/MeasureValue,LabSamplePreparationUrl
count,0.0,0.0,0.0,6089.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,320.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,44066.000000,0.0
mean,NaN,NaN,NaN,1.005012,NaN,NaN,NaN,NaN,NaN,NaN,...,29901.318750,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.813667,NaN
std,NaN,NaN,NaN,2.617023,NaN,NaN,NaN,NaN,NaN,NaN,...,34542.438576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,156.132388,NaN
min,NaN,NaN,NaN,0.100000,NaN,NaN,NaN,NaN,NaN,NaN,...,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.000000,NaN
25%,NaN,NaN,NaN,0.100000,NaN,NaN,NaN,NaN,NaN,NaN,...,63.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.050000,NaN
50%,NaN,NaN,NaN,0.100000,NaN,NaN,NaN,NaN,NaN,NaN,...,452.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,NaN
75%,NaN,NaN,NaN,0.300000,NaN,NaN,NaN,NaN,NaN,NaN,...,68498.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN
max,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,91986.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000.000000,NaN


# Step 2: Rename Columns

In [8]:
# Lowercase, strip spaces, replace gaps with underscores
df_wq.columns = df_wq.columns.str.strip().str.replace(" ", "_")
df_wq.columns

Index(['OrganizationIdentifier', 'OrganizationFormalName',
       'ActivityIdentifier', 'ActivityTypeCode', 'ActivityMediaName',
       'ActivityMediaSubdivisionName', 'ActivityStartDate',
       'ActivityStartTime/Time', 'ActivityStartTime/TimeZoneCode',
       'ActivityEndDate', 'ActivityEndTime/Time',
       'ActivityEndTime/TimeZoneCode', 'ActivityRelativeDepthName',
       'ActivityDepthHeightMeasure/MeasureValue',
       'ActivityDepthHeightMeasure/MeasureUnitCode',
       'ActivityDepthAltitudeReferencePointText',
       'ActivityTopDepthHeightMeasure/MeasureValue',
       'ActivityTopDepthHeightMeasure/MeasureUnitCode',
       'ActivityBottomDepthHeightMeasure/MeasureValue',
       'ActivityBottomDepthHeightMeasure/MeasureUnitCode', 'ProjectIdentifier',
       'ProjectName', 'ActivityConductingOrganizationText',
       'MonitoringLocationIdentifier', 'MonitoringLocationName',
       'ActivityCommentText', 'SampleAquifer', 'HydrologicCondition',
       'HydrologicEvent', 'Activi

# Step 3: Delete Trivial Columns
Or columns with too many missing values

In [9]:
# Find percent of missing values in each column

percent_missing = df_wq.isnull().sum() * 100 / len(df_wq)
missing_value_df = pd.DataFrame({'column_name': df_wq.columns,
                                 'percent_missing': percent_missing})
missing_value_df.head(20)

,column_name,percent_missing
OrganizationIdentifier,OrganizationIdentifier,0.000000
OrganizationFormalName,OrganizationFormalName,0.000000
ActivityIdentifier,ActivityIdentifier,0.000000
ActivityTypeCode,ActivityTypeCode,0.000000
ActivityMediaName,ActivityMediaName,0.000000
ActivityMediaSubdivisionName,ActivityMediaSubdivisionName,52.533641
ActivityStartDate,ActivityStartDate,0.000000
ActivityStartTime/Time,ActivityStartTime/Time,1.113790
ActivityStartTime/TimeZoneCode,ActivityStartTime/TimeZoneCode,1.113790
ActivityEndDate,ActivityEndDate,100.000000


In [10]:
# Replace empty strings and whitespace-only strings with NaN
df_wq.replace(r'^\s*$', pd.NA, regex=True, inplace=True)

# Drop columns where more than 90% of values are missing (NaN/None/pd.NA)
threshold = len(df_wq) * 0.10 # Minimum number of non-missing values required to keep column
df_wq.dropna(axis=1, thresh=threshold, inplace=True)

# Recalculate percent of missing values in each column

percent_missing = df_wq.isnull().sum() * 100 / len(df_wq)
missing_value_df = pd.DataFrame({'column_name': df_wq.columns,
                                 'percent_missing': percent_missing})
missing_value_df

,column_name,percent_missing
OrganizationIdentifier,OrganizationIdentifier,0.000000
OrganizationFormalName,OrganizationFormalName,0.000000
ActivityIdentifier,ActivityIdentifier,0.000000
ActivityTypeCode,ActivityTypeCode,0.000000
ActivityMediaName,ActivityMediaName,0.000000
ActivityMediaSubdivisionName,ActivityMediaSubdivisionName,52.533641
ActivityStartDate,ActivityStartDate,0.000000
ActivityStartTime/Time,ActivityStartTime/Time,1.113790
ActivityStartTime/TimeZoneCode,ActivityStartTime/TimeZoneCode,1.113790
ProjectIdentifier,ProjectIdentifier,0.037098


In [41]:
# Re-create the df_wq data frame to include only necessary columns

df_wq = df_wq[[
 "ActivityIdentifier", 
 "ActivityStartDate", 
 "ActivityStartTime/Time",  
 "MonitoringLocationIdentifier", 
 "MonitoringLocationName", 
 "ActivityLocation/LatitudeMeasure", 
 "ActivityLocation/LongitudeMeasure",
 "CharacteristicName",
 "ResultMeasureValue",
 "ResultMeasure/MeasureUnitCode",
 "ResultValueTypeName",
 "AnalysisStartDate",
 "LastUpdated"
]]

# Step 4: Fix Data Types

Print data types of existing columns

In [42]:
#for i, t in enumerate(df_wq.dtypes):
 #   print(f'{t}: {df_wq.columns[i]}')
df_wq.dtypes

ActivityIdentifier                              str
ActivityStartDate                               str
ActivityStartTime/Time                          str
MonitoringLocationIdentifier                    str
MonitoringLocationName                          str
ActivityLocation/LatitudeMeasure            float64
ActivityLocation/LongitudeMeasure           float64
CharacteristicName                              str
ResultMeasureValue                          float64
ResultMeasure/MeasureUnitCode                   str
ResultValueTypeName                             str
AnalysisStartDate                    datetime64[us]
LastUpdated                                     str
dtype: object

In [ ]:
# Type conversions are handled in the cell below

In [ ]:
# Combine date + time into a single datetime column
df_wq["ActivityStartDateTime"] = df_wq["ActivityStartDate"] + " " + df_wq["ActivityStartTime/Time"]
df_wq["ActivityStartDateTime"] = pd.to_datetime(df_wq["ActivityStartDateTime"], errors="coerce")
df_wq.drop(["ActivityStartDate", "ActivityStartTime/Time"], axis=1, inplace=True)

# Convert remaining date columns to datetime
df_wq["AnalysisStartDate"] = pd.to_datetime(df_wq["AnalysisStartDate"], errors="coerce")
df_wq["LastUpdated"] = pd.to_datetime(df_wq["LastUpdated"], errors="coerce")

# Convert measurement value to float
df_wq["ResultMeasureValue"] = pd.to_numeric(df_wq["ResultMeasureValue"], errors="coerce")

df_wq.dtypes

In [45]:
df_wq.dtypes

ActivityIdentifier                              str
ActivityStartDate                               str
ActivityStartTime/Time                          str
MonitoringLocationIdentifier                    str
MonitoringLocationName                          str
ActivityLocation/LatitudeMeasure            float64
ActivityLocation/LongitudeMeasure           float64
CharacteristicName                              str
ResultMeasureValue                          float64
ResultMeasure/MeasureUnitCode                   str
ResultValueTypeName                             str
AnalysisStartDate                    datetime64[us]
LastUpdated                                     str
ActivityStartDateTime                datetime64[us]
dtype: object

In [ ]:
# Step 5: Handle Missing Values
# Drop rows missing the key columns needed for analysis
df_wq.dropna(subset=["MonitoringLocationIdentifier", "CharacteristicName", "ResultMeasureValue"], inplace=True)

print(f"Rows remaining: {len(df_wq):,}")
print("\nMissing values per column:")
print(df_wq.isnull().sum())

In [ ]:
# Step 6: Save Cleaned Data
import os
out_path = '../../../../data/tabular/water-quality/cleaned'
os.makedirs(out_path, exist_ok=True)
df_wq.to_csv(f'{out_path}/epa-wq-clean.csv', index=False)
print(f"Saved {len(df_wq):,} rows → {out_path}/epa-wq-clean.csv")